In [ ]:
import sys
sys.path.append("/project/src")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines import KaplanMeierFitter
from sksurv.metrics import integrated_brier_score, cumulative_dynamic_auc, brier_score
from sklearn.metrics import mean_squared_error, roc_curve
from lifelines.statistics import multivariate_logrank_test, pairwise_logrank_test

from preprocessing import (
    split_features_target,
    low_missingness_complete_case_analysis,
    SURVIVAL_EVENT_COL,
    SURVIVAL_TIME_COL,
)
from helpers import (
  save_pic,
  show_pic
)

import joblib

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    train_df_csv = "/content/drive/MyDrive/bachelor/nacc_train_reduced.csv"
    test_df_csv = "/content/drive/MyDrive/bachelor/nacc_test.csv"
else:
    train_df_csv = "./data/nacc_train_reduced.csv"
    test_df_csv = "./data/nacc_test.csv"

In [ ]:
train_df = pd.read_csv(train_df_csv, delimiter=',')
test_df = pd.read_csv(test_df_csv, delimiter=',')

In [ ]:
train_X, train_y = split_features_target(train_df)

In [ ]:
plt.figure(figsize=(15, 8))
bars = sns.barplot(x=test_df['EVENT_MCI'].value_counts().index, y=test_df['EVENT_MCI'].value_counts().values)
for bar in bars.containers:
    ax = bars.axes
    ax.bar_label(bar, padding=3, fontsize=8)
plt.title('Distribution of EVENT_MCI')
plt.xlabel('EVENT_MCI')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
test_df['EVENT_MCI'].value_counts(normalize=True)

# Cross-Validation results

In [ ]:
rsf_pipeline = joblib.load("joblib-storage/rsf_best_pipeline.joblib")
preprocessor = rsf_pipeline.named_steps['preprocessor']
rsf = rsf_pipeline.named_steps['model']

In [ ]:
rsf_best_params = joblib.load("joblib-storage/rsf_best_params.joblib")
print(f"Best RSF parameters: {rsf_best_params}")

In [ ]:
cv_cindex = joblib.load("joblib-storage/rsf_best_cv_cindex.joblib")
cv_cindex_std = joblib.load("joblib-storage/rsf_best_cv_cindex_std.joblib")
print(f"Best CV C-index: {cv_cindex:.4f} ± {cv_cindex_std:.4f}")

# Evaluaition on test dataset

In [ ]:
fitted_low_missingness_cols = preprocessor.get_low_missingness_cols()
test_preprepared = low_missingness_complete_case_analysis(test_df, low_missingness_columns=fitted_low_missingness_cols)
test_X_raw, test_y = split_features_target(test_preprepared)
test_X = preprocessor.transform(test_X_raw)

In [ ]:
test_events = test_y[SURVIVAL_EVENT_COL].astype(bool)
test_times = test_y[SURVIVAL_TIME_COL]

## Test c-index

In [ ]:
cindex_test = rsf.score(test_X, test_y)
print(f"Test C-index: {cindex_test:.4f}")

## Test IBS

In [ ]:
def calculate_ibs(cox, test_X, test_y, event_min_time, event_max_time):
  times_ibs = np.linspace(event_min_time, event_max_time, 100)
  # RSF returns (surv_matrix, unique_death_times), interpolate to get survival probs at specific times
  surv_matrix_raw, death_times = rsf.predict_survival_function(test_X)

  # Interpolate each patient's survival curve to times_ibs
  surv_matrix = np.array([
      np.interp(times_ibs, death_times, surv_matrix_raw[i])
      for i in range(surv_matrix_raw.shape[0])
  ])

  ibs = integrated_brier_score(train_y, test_y, surv_matrix, times_ibs)
  print(f"Integrated Brier Score: {ibs:.4f}")
  return ibs, surv_matrix_raw, death_times

In [ ]:
event_min_time = test_times[test_events].min()
event_max_time = test_times[test_events].max()
print(f"Event times range from {event_min_time:.2f} to {event_max_time:.2f} years")

In [ ]:
follow_up_ibs, surv_matrix_raw, death_times = calculate_ibs(rsf, test_X, test_y, event_min_time, event_max_time)

In [ ]:
event_min_time = test_times[test_events].min()
event_max_time = 5
print(f"Event times range from {event_min_time:.2f} to {event_max_time:.2f} years")

ibs_5_years, _, _ = calculate_ibs(rsf, test_X, test_y, event_min_time, event_max_time)

## Test KM and Survival curve

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(test_times, event_observed=test_events)

In [ ]:
# Mean predicted survival curve across all patients
predicted_mean_surv = surv_matrix_raw.mean(axis=0)
predicted_surv = pd.Series(predicted_mean_surv, index=death_times)
predicted_surv.head(5)

In [ ]:
test_times_min = test_times.min()
test_times_max = test_times.max()
times_to_plot = np.linspace(test_times_min, test_times_max, 100)

In [ ]:
brier_times = np.arange(int(test_times_min)+1, int(test_times_max)+1)

# Interpolate survival curves to integer year time points for Brier score
brier_surv_matrix = np.array([
    np.interp(brier_times, death_times, surv_matrix_raw[i])
    for i in range(surv_matrix_raw.shape[0])
])

_, brier_scores = brier_score(train_y, test_y, brier_surv_matrix, brier_times)
brier_scores

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
kmf.plot_survival_function(ax=ax[0], label="Kaplan-Meier (observed)", color="steelblue")
ax[0].plot(predicted_surv.index, predicted_surv.values, color="tomato", linewidth=2, label="RSF model (predicted)")
ax[0].set_xlabel("Time (years)", fontsize=12)
ax[0].set_ylabel("Survival probability", fontsize=12)
ax[0].set_title("Kaplan-Meier vs RSF model predicted survival curves", fontsize=14)
ax[0].legend()  

ax[1].plot(brier_times, brier_scores, color="green", marker="o", linewidth=2)
ax[1].axhline(0.25, color="red", linestyle="--", label=f"Integrated Brier Score = {follow_up_ibs:.4f}")
ax[1].set_xlabel("Time (years)", fontsize=12)
ax[1].set_ylabel("Brier Score", fontsize=12)
ax[1].set_title("Brier Score over time", fontsize=14)
ax[1].legend()
plt.tight_layout()
save_pic(plt, "rsf_evaluation_KM_IBS.png")
plt.show()

In [ ]:
times = np.linspace(test_times_min, test_times_max, 100)

# Interpolate survival curves to evaluation times
pred_surv = np.array([
    np.interp(times, death_times, surv_matrix_raw[i])
    for i in range(surv_matrix_raw.shape[0])
])
predicted_mean = np.mean(pred_surv, axis=0)
kmf_surv_scores = kmf.survival_function_at_times(times).values

abs_errors = np.abs(kmf_surv_scores - predicted_mean)

rmse      = np.sqrt(mean_squared_error(kmf_surv_scores, predicted_mean))
median_ae = np.median(abs_errors)
mean_ae   = np.mean(abs_errors)

print(f"RMSE:             {rmse:.4f}")
print(f"Median Abs Error: {median_ae:.4f}")
print(f"Mean Abs Error:   {mean_ae:.4f}")

## Risk groups

In [ ]:
LOW_RISK_GROUP_INDICATOR = 0
MEDIUM_RISK_GROUP_INDICATOR = 1
HIGH_RISK_GROUP_INDICATOR = 2

def assign_patient_to_risk_group(score, low_cutoff, high_cutoff):
    if score <= low_cutoff:
        return LOW_RISK_GROUP_INDICATOR
    elif score <= high_cutoff:
        return MEDIUM_RISK_GROUP_INDICATOR
    else:
        return HIGH_RISK_GROUP_INDICATOR

In [ ]:
times_curve = np.linspace(test_times.min(), test_times.max(), 100)
conv_curve = np.array([
    np.interp(times_curve, death_times, 1 - surv_matrix_raw[i])
    for i in range(surv_matrix_raw.shape[0])
])
density_score_per_patient = conv_curve.mean(axis=1)

low_medium_cutoff  = np.percentile(density_score_per_patient, 33.333)
medium_high_cutoff = np.percentile(density_score_per_patient, 66.667)

risk_df = pd.DataFrame({
    "scores":     density_score_per_patient,
    "event":      test_y[SURVIVAL_EVENT_COL].astype(bool),
    "duration":   test_y[SURVIVAL_TIME_COL],
})
risk_df["risk_group"] = risk_df["scores"].apply(
    lambda s: assign_patient_to_risk_group(s, low_medium_cutoff, medium_high_cutoff)
)

print(f"Cutoffs: low/medium={low_medium_cutoff:.3f}, medium/high={medium_high_cutoff:.3f}")
print(f"Low risk group:    {(risk_df['risk_group'] == LOW_RISK_GROUP_INDICATOR).sum()} patients")
print(f"Medium risk group: {(risk_df['risk_group'] == MEDIUM_RISK_GROUP_INDICATOR).sum()} patients")
print(f"High risk group:   {(risk_df['risk_group'] == HIGH_RISK_GROUP_INDICATOR).sum()} patients")

In [ ]:
NUMBER_OF_BINS = 80
bin_edges = np.linspace(risk_df["scores"].min(), risk_df["scores"].max(), NUMBER_OF_BINS + 1)

low_scores    = risk_df.loc[risk_df["risk_group"] == LOW_RISK_GROUP_INDICATOR, "scores"].values
medium_scores = risk_df.loc[risk_df["risk_group"] == MEDIUM_RISK_GROUP_INDICATOR, "scores"].values
high_scores   = risk_df.loc[risk_df["risk_group"] == HIGH_RISK_GROUP_INDICATOR, "scores"].values

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(low_scores,    bins=bin_edges, color="green",  label=f"Low Risk (n={len(low_scores)})",    edgecolor="white", alpha=0.7)
ax.hist(medium_scores, bins=bin_edges, color="orange", label=f"Medium Risk (n={len(medium_scores)})", edgecolor="white", alpha=0.7)
ax.hist(high_scores,   bins=bin_edges, color="red",    label=f"High Risk (n={len(high_scores)})",   edgecolor="white", alpha=0.7)

ax.axvline(low_medium_cutoff,  color='black', linestyle='--', linewidth=1, alpha=0.6)
ax.axvline(medium_high_cutoff, color='black', linestyle='--', linewidth=1, alpha=0.6)
ymax = ax.get_ylim()[1]
ax.text(low_medium_cutoff + 0.005,  ymax * 0.92, f'{low_medium_cutoff:.3f}',  fontsize=9)
ax.text(medium_high_cutoff + 0.005, ymax * 0.92, f'{medium_high_cutoff:.3f}', fontsize=9)

ax.set_xlabel("Mean predicted conversion probability", fontsize=10)
ax.set_ylabel("Number of Patients", fontsize=10)
ax.set_title("RSF: Risk group distribution (all patients)", fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
save_pic(plt, "rsf_evaluation_Risk_Groups.png")
plt.show()

In [ ]:
print(f"Predicted conversion probability at end of follow-up ({times_curve[-1]:.2f} years):")
group_risk_probs = {}
for grp, name in [(LOW_RISK_GROUP_INDICATOR, "Low"),
                  (MEDIUM_RISK_GROUP_INDICATOR, "Medium"),
                  (HIGH_RISK_GROUP_INDICATOR, "High")]:
    mask = (risk_df["risk_group"] == grp).values
    prob = conv_curve[mask].mean(axis=0)[-1] * 100
    group_risk_probs[grp] = prob
    print(f"  {name:<7} risk: {prob:.1f}%  (n={mask.sum()})")

In [ ]:
overall = multivariate_logrank_test(
    risk_df['duration'],
    risk_df['risk_group'],
    risk_df['event']
)

fig, ax = plt.subplots(figsize=(9, 6))

risk_groups_props = {
    LOW_RISK_GROUP_INDICATOR:    {"color": 'green',  "name": "Low risk"},
    MEDIUM_RISK_GROUP_INDICATOR: {"color": 'orange', "name": "Medium risk"},
    HIGH_RISK_GROUP_INDICATOR:   {"color": 'red',    "name": "High risk"}
}

# conv_curve shape: (n_patients, n_timepoints), computed in the risk group cell
surv_curve = 1 - conv_curve

for group in [LOW_RISK_GROUP_INDICATOR, MEDIUM_RISK_GROUP_INDICATOR, HIGH_RISK_GROUP_INDICATOR]:
    mask = (risk_df['risk_group'] == group).values
    mean_surv = surv_curve[mask].mean(axis=0)
    ax.plot(times_curve, mean_surv,
            color=risk_groups_props[group]["color"], linewidth=2,
            label=f'{risk_groups_props[group]["name"]} (n={mask.sum()}, conversion prob={group_risk_probs[group]:.2f}%)')

ax.set_xlabel("Time (years)")
ax.set_ylabel("Survival probability")
ax.set_title(f"RSF: Predicted survival curves by risk group  |  log-rank p={overall.p_value:.2e}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
save_pic(plt, "rsf_evaluation_Risk_Groups_KM.png")
plt.show()

In [ ]:
overall = multivariate_logrank_test(
    risk_df['duration'],
    risk_df['risk_group'],
    risk_df['event']
)

fig, ax = plt.subplots(figsize=(9, 6))

for group in [LOW_RISK_GROUP_INDICATOR, MEDIUM_RISK_GROUP_INDICATOR, HIGH_RISK_GROUP_INDICATOR]:
    mask = (risk_df['risk_group'] == group).values
    mean_conv = conv_curve[mask].mean(axis=0) * 100   # CDF in %
    ax.plot(times_curve, mean_conv,
            color=risk_groups_props[group]["color"], linewidth=2,
            label=f'{risk_groups_props[group]["name"]} (n={mask.sum()}, conversion prob={group_risk_probs[group]:.2f}%)')

ax.set_xlabel("Time (years)")
ax.set_ylabel("Probability of conversion (%)")
ax.set_title(f"RSF: Predicted CDF of conversion by risk group  |  log-rank p={overall.p_value:.2e}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
save_pic(plt, "rsf_evaluation_Risk_Groups_CDF.png")
plt.show()

### Analyse significance of group separation

In [ ]:
overall = multivariate_logrank_test(
    risk_df['duration'],
    risk_df['risk_group'],
    risk_df['event']
)
overall.print_summary()
print(f"Overall log-rank p-value: {overall.p_value:.20f}")

pairwise = pairwise_logrank_test(
    risk_df['duration'],
    risk_df['risk_group'],
    risk_df['event']
)
pairwise.print_summary()

## Time dependent ROC-AUC

In [ ]:
times_roc   = np.arange(1, 11)
risk_scores = rsf.predict(test_X)
auc_scores, mean_auc = cumulative_dynamic_auc(train_y, test_y, risk_scores, times_roc)

plt.plot(times_roc, auc_scores, marker='o', color='steelblue')
plt.xlabel("Time (years)")
plt.ylabel("AUC")
plt.title("RSF: Time-dependent AUC")
plt.grid(True, alpha=0.2)

plt.show()

for t, a in zip(times_roc, auc_scores):
    print(f"AUC at {t:.0f} year(s): {a:.4f}")
print(f"Mean AUC:          {mean_auc:.4f}")

In [ ]:
times_roc   = np.arange(int(test_times_min)+1, int(test_times_max) + 1)
risk_scores = rsf.predict(test_X)
auc_scores, mean_auc = cumulative_dynamic_auc(train_y, test_y, risk_scores, times_roc)

plt.plot(times_roc, auc_scores, marker='o', color='steelblue')
plt.xlabel("Time (years)")
plt.ylabel("AUC")
plt.title("RSF: Time-dependent AUC")
plt.grid(True, alpha=0.2)

plt.show()

for t, a in zip(times_roc, auc_scores):
    print(f"AUC at {t:.0f} year(s): {a:.4f}")
print(f"Mean AUC:          {mean_auc:.4f}")

In [ ]:
for t in times_roc:
  cases = (test_times >= t) & (test_events == True)
  controls = test_times >= t
  cases_num = cases.sum()
  controls_num = controls.sum()
  print(f"At time {t:.0f} years: {cases_num} cases, {controls_num} controls, {cases_num/controls_num:.2f} cases/controls ratio")

In [ ]:
times_roc = [1, 3, 5]
print(f"Using timepoints: {times_roc}")

risk_scores_roc = rsf.predict(test_X)
auc_scores, mean_auc = cumulative_dynamic_auc(
    train_y, test_y, risk_scores_roc, times_roc
)
print(f"AUC scores: {auc_scores}")

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['steelblue', 'tomato', 'green']

for t, auc_val, color in zip(times_roc, auc_scores, colors):
    known_mask    = (test_y[SURVIVAL_EVENT_COL].astype(bool)) | (test_y[SURVIVAL_TIME_COL] >= t)
    y_true_binary = ((test_y[SURVIVAL_EVENT_COL].astype(bool)) & (test_y[SURVIVAL_TIME_COL] <= t))[known_mask].astype(int)
    risk_known    = risk_scores_roc[known_mask]

    fpr, tpr, _   = roc_curve(y_true_binary, risk_known)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{t} year(s) AUC = {auc_val:.3f}')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(f"RSF: Time-dependent ROC  |  Mean AUC = {mean_auc:.3f}",
             fontsize=12)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
save_pic(plt, "rsf_evaluation_ROC_AUCs.png")
plt.show()

# Summary results

## CV C-index

In [ ]:
print(f"cross-validation C-index: {cv_cindex:.2f} ± {cv_cindex_std:.2f}")

## Test C-index

In [ ]:
print(f"Test C-index: {cindex_test:.2f}")

## IBS score

In [ ]:
print(f"IBS score for the entire follow-up period: {follow_up_ibs:.3f}")
print(f"IBS score for the first 5 years: {ibs_5_years:.3f}")

## KM curve

In [ ]:
show_pic("rsf_evaluation_KM_IBS.png")

In [ ]:
print(f"RMSE:             {rmse:.4f}")
print(f"Median Abs Error: {median_ae:.4f}")
print(f"Mean Abs Error:   {mean_ae:.4f}")

## Risk groups

In [ ]:
show_pic("rsf_evaluation_Risk_Groups.png")

In [ ]:
show_pic("rsf_evaluation_Risk_Groups_KM.png")

In [ ]:
for gr in group_risk_probs:
    print(f"{risk_groups_props[gr]['name']} conversion probability: {group_risk_probs[gr]:.2f}%")

### Risk groups long-rank tests results

In [ ]:
pairwise.print_summary()

In [ ]:
overall.print_summary()

In [ ]:

risk_df.groupby('risk_group')['scores'].mean().rename(index={0: 'low', 1: 'medium', 2: 'high'})


## Integrated ROC-AUC

In [ ]:
print(f"Using timepoints (in years): {times_roc}")
print(f"AUC scores: {np.round(auc_scores, 3)}")

In [ ]:
show_pic("rsf_evaluation_ROC_AUCs.png")